# VLM-Anomaly — Full MVTec Sweep · Groq Free Tier (All 15 Categories)

**Zero cost. Zero data download. Just your GROQ_API_KEY.**

## Setup (do this once before running)
1. **Add Groq key** → Notebook → Add-ons → Secrets → `GROQ_API_KEY`
2. **Attach MVTec dataset** → Notebook → Input → Search *"mvtec-ad"* → Add `ipythonx/mvtec-ad`
3. Click **Run All**

Results are saved to `/kaggle/working/results/` and zipped for download.
Expected runtime: **60–90 minutes** (83 images × 15 categories = 1,245 Groq calls, free tier).


In [ ]:
# ── Cell 1: Install dependencies from PyPI ──────────────────────────────────
# vlm_anomaly source is attached as the dataset: sabareeswarans11/vlm-anomaly-src
# All other deps are standard PyPI packages — internet must be ON.
import subprocess, sys, os

# Add vlm_anomaly from the attached Kaggle dataset to Python path
VLM_SRC = "/kaggle/input/vlm-anomaly-src/src"
if VLM_SRC not in sys.path:
    sys.path.insert(0, VLM_SRC)

# Install only standard pip dependencies (no git clone needed)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "httpx>=0.28", "tenacity", "pyyaml",
    "pydantic>=2.10", "pydantic-settings>=2.6",
    "structlog", "Pillow", "duckdb",
    "scikit-learn", "tqdm", "tabulate",
])

# Verify import works
import vlm_anomaly
print(f"✓ vlm_anomaly {vlm_anomaly.__version__} loaded from {VLM_SRC}")

In [ ]:
# ── Cell 2: Groq API key from Kaggle Secrets ────────────────────────────────
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
groq_key = secrets.get_secret("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = groq_key
print(f"✓ GROQ_API_KEY set (length={len(groq_key)})")

In [ ]:
# ── Cell 3: Paths & dataset check ───────────────────────────────────────────
from pathlib import Path

# Kaggle mounts ipythonx/mvtec-ad at /kaggle/input/mvtec-ad/
MVTEC_ROOT  = Path("/kaggle/input/mvtec-ad")
RESULTS_DIR = Path("/kaggle/working/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert MVTEC_ROOT.exists(), f"Attach the 'ipythonx/mvtec-ad' dataset in the sidebar!"

categories = sorted(p.name for p in MVTEC_ROOT.iterdir()
                    if p.is_dir() and not p.name.startswith("."))
print(f"✓ MVTec root   : {MVTEC_ROOT}")
print(f"✓ Results dir  : {RESULTS_DIR}")
print(f"✓ Categories   : {len(categories)} found")
print(f"   {categories}")

In [ ]:
# ── Cell 4: Configure ───────────────────────────────────────────────────────
# Edit here if you want to change anything.

PROMPT_KEY  = "manufacturing.detailed"   # best prompt for MVTec
LIMIT       = None                        # None = all images; set e.g. 10 for a quick test
BUDGET_USD  = 100.0                       # Groq is free, but keeps the evaluator happy
MODEL       = "meta-llama/llama-4-scout-17b-16e-instruct"

print(f"Prompt : {PROMPT_KEY}")
print(f"Limit  : {LIMIT or 'all images'}")
print(f"Model  : {MODEL}")
print(f"Total  : ~{len(categories) * 83} images (83 avg per category)")

In [ ]:
# ── Cell 5: Build shared objects ────────────────────────────────────────────
from vlm_anomaly.config import Settings
from vlm_anomaly.datasets.mvtec import MVTec
from vlm_anomaly.backends.groq import GroqBackend
from vlm_anomaly.evaluators.prompt_library import PromptLibrary
from vlm_anomaly.logging import configure_logging

configure_logging(json_logs=False, log_level="INFO")

settings = Settings(
    _env_file="/dev/null",
    data_dir=str(MVTEC_ROOT.parent),
    results_dir=str(RESULTS_DIR),
    default_budget_usd=BUDGET_USD,
)
# Override data_dir so MVTec loader finds /kaggle/input/mvtec-ad/
import vlm_anomaly.datasets.mvtec as _m
dataset = MVTec(root_dir=MVTEC_ROOT)
backend = GroqBackend(model=MODEL)
prompt_lib = PromptLibrary(prompts_dir="/kaggle/input/vlm-anomaly-src/prompts")

print(f"✓ Dataset   : {dataset.root_dir}")
print(f"✓ Backend   : {backend.name} / {MODEL}")
print(f"✓ Prompts   : {len(prompt_lib.available_keys())} keys loaded")

In [ ]:
# ── Cell 6: Run all 15 categories ────────────────────────────────────────────
from tqdm.notebook import tqdm
from vlm_anomaly.schemas import ExperimentConfig
from vlm_anomaly.evaluators.vlm_evaluator import VLMEvaluator

all_results = []

for category in tqdm(categories, desc="MVTec categories"):
    # Idempotency: skip if non-empty JSONL already exists for this category
    existing = [f for f in RESULTS_DIR.glob(f"*_mvtec_{category}.jsonl")
                if f.stat().st_size > 100]
    if existing:
        print(f"  [skip] {category} — already done ({existing[0].name})")
        continue

    config = ExperimentConfig(
        backend="groq",
        dataset="mvtec",
        categories=[category],
        prompt=PROMPT_KEY,
        limit=LIMIT,
        budget_usd=BUDGET_USD,
    )
    evaluator = VLMEvaluator(
        backend=backend,
        dataset=dataset,
        config=config,
        settings=settings,
        prompt_library=prompt_lib,
    )
    results = evaluator.run()
    all_results.extend(results)

    for r in results:
        auroc = f"{r.auroc:.3f}" if r.auroc is not None else " N/A"
        f1    = f"{r.f1:.3f}"   if r.f1    is not None else " N/A"
        print(f"  ✓ {r.category:<15}  n={r.n_images:3d}  auroc={auroc}  f1={f1}  cost=${r.total_cost_usd:.4f}")

print(f"\nAll done. {len(all_results)} categories evaluated.")

In [ ]:
# ── Cell 7: Leaderboard ─────────────────────────────────────────────────────
from vlm_anomaly.analysis.aggregator import leaderboard, cost_accuracy_table

lb = leaderboard(RESULTS_DIR)
if lb.empty:
    print("No results yet.")
else:
    summary = cost_accuracy_table(RESULTS_DIR)
    print("=== Summary (mean across categories) ===")
    print(summary[["model_id","mean_auroc","mean_latency_ms"]].to_string(index=False))
    print()
    print("=== Per-category breakdown ===")
    display(lb[["category","n_images","auroc","f1","mean_latency_ms"]].sort_values("auroc", ascending=False))

In [ ]:
# ── Cell 8: Generate report ──────────────────────────────────────────────────
from vlm_anomaly.analysis.report_generator import generate

report = generate(RESULTS_DIR, "/kaggle/working/REPORT.md")
print(f"✓ Report written to {report}")

plots = list((RESULTS_DIR / "plots").glob("*.png"))
print(f"✓ Plots: {[p.name for p in plots]}")

In [ ]:
# ── Cell 9: Zip results for download ─────────────────────────────────────────
import shutil

out = shutil.make_archive("/kaggle/working/vlm_anomaly_groq_results", "zip", RESULTS_DIR)
print(f"✓ Download: {out}")
print()
print("After downloading, commit locally:")
print("  unzip vlm_anomaly_groq_results.zip -d results/")
print("  git add results/*.jsonl")
print("  git commit -m 'results: Groq Llama-4-Scout full MVTec sweep (15 categories)'")
print("  git push")